In [4]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('german_credit.db')
cursor = conn.cursor()

X = pd.read_csv('features_encoded.csv')
y = pd.read_csv('targets.csv') 


columns = X.columns.tolist()
columns.append('Creditworthiness')

create_table_query = f"""
CREATE TABLE IF NOT EXISTS german_credit (
    {", ".join([f'[{col}] TEXT' for col in columns])}
);
"""

column_defs = [f'[{col}] REAL' for col in columns]
column_defs_str = ",\n    ".join(column_defs)

create_table_query = f"""
CREATE TABLE IF NOT EXISTS german_credit (
    {column_defs_str}
);
"""

cursor.execute(create_table_query)
conn.commit()

df_to_db = X.copy()
df_to_db['Creditworthiness'] = y.iloc[:, -1] 


df_to_db.to_sql('german_credit', conn, if_exists='replace', index=False)

conn.close() 

In [9]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('german_credit.db')

def run_query(title, query):
    print(f" {title} ")
    display(pd.read_sql(query, conn))
    print("\n")

In [10]:
q1 = """
SELECT 
    Age, 
    Duration, 
    "Credit amount"
FROM german_credit
ORDER BY "Credit amount" DESC
LIMIT 5;
"""
run_query("Топ-5 самых крупных кредитов", q1)

 Топ-5 самых крупных кредитов 


,Age,Duration,Credit amount
0,32,48,18424
1,58,54,15945
2,43,36,15857
3,23,48,15672
4,21,60,15653


In [11]:
q2 = """
SELECT 
    COUNT(*) as "Total Clients",
    ROUND(AVG(Age), 1) as "Average Age",
    ROUND(AVG("Credit amount"), 2) as "Average Credit",
    ROUND(AVG(Duration), 1) as "Average Duration (Months)",
    SUM("Credit amount") as "Total Portfolio Value"
FROM german_credit;
"""
run_query("Общая статистика портфеля", q2)

 Общая статистика портфеля 


,Total Clients,Average Age,Average Credit,Average Duration (Months),Total Portfolio Value
0,1000,35.5,3271.26,20.9,3271258


In [12]:
q3 = """
SELECT 
    CASE 
        WHEN Age < 30 THEN 'Young (<30)'
        WHEN Age BETWEEN 30 AND 50 THEN 'Adult (30-50)'
        ELSE 'Senior (>50)'
    END as Age_Group,
    COUNT(*) as Count,
    ROUND(AVG("Credit amount"), 0) as "Avg Credit Amount"
FROM german_credit
GROUP BY Age_Group
ORDER BY "Avg Credit Amount" DESC;
"""
run_query("Анализ по возрастным группам", q3)

 Анализ по возрастным группам 


,Age_Group,Count,Avg Credit Amount
0,Adult (30-50),516,3397.0
1,Senior (>50),113,3297.0
2,Young (<30),371,3089.0


In [13]:
q4 = """
SELECT 
    CASE 
        WHEN "Housing_A152" = 1.0 THEN 'Owner'
        ELSE 'Non-Owner'
    END as Housing_Status,
    COUNT(*) as Total_People,
    ROUND(AVG("Credit amount"), 0) as Avg_Credit,
    ROUND(AVG(Duration), 1) as Avg_Duration
FROM german_credit
GROUP BY Housing_Status;
"""
run_query("Сравнение собственников жилья и остальных", q4)

 Сравнение собственников жилья и остальных 


,Housing_Status,Total_People,Avg_Credit,Avg_Duration
0,Non-Owner,287,3794.0,22.3
1,Owner,713,3061.0,20.3


In [15]:
conn.close()